# XRF BCF Element Extraction

Converts raw Bruker `.bcf` hypercubes into per-element `_raw.tiff` files
using dynamic emission-line resolution and dual-window Bremsstrahlung
subtraction.

**Outputs** land in `data/xrf/raw/` following the schema
`{sample_name}_{element_line}_raw.tiff`.

In [ ]:
from pathlib import Path

from xrf import Bcf_Element_Extractor
from xrf.config import Bcf_Extraction_Config

## 1.  Path constants

Set the input BCF directory and the output `raw` directory relative to the
project root.

In [ ]:
PROJECT_ROOT = Path.cwd()
BCF_INPUT_DIR = PROJECT_ROOT / "data" / "xrf" / "bcf"
RAW_OUTPUT_DIR = PROJECT_ROOT / "data" / "xrf" / "raw"

print(f"BCF input  : {BCF_INPUT_DIR}")
print(f"TIFF output: {RAW_OUTPUT_DIR}")

## 2.  Discover `.bcf` files

List every `.bcf` hypercube present in the input directory.

In [ ]:
bcf_files = sorted(BCF_INPUT_DIR.glob("*.bcf"))
print(f"Found {len(bcf_files)} BCF file(s):")
for bcf_file in bcf_files:
    print(f"  {bcf_file.name}")

## 3.  Element selection

Choose target elements by **symbol string only** — the extractor resolves the
correct emission line automatically.
Elements Z ≤ 40 default to Kα; heavier elements default to Lα.
Append an explicit line (e.g. `"Au_La"`) to force a specific transition.

In [ ]:
TARGET_ELEMENTS = ["Fe", "Cu", "Au", "Hg", "Pb", "Ca", "K", "As"]
print("Target elements:")
for elem in TARGET_ELEMENTS:
    print(f"  {elem}")

## 4.  Configure & instantiate the extractor

The `Bcf_Extraction_Config` dataclass holds all extraction hyperparameters.
Pass it to `Bcf_Element_Extractor` or instantiate with keyword overrides.

In [ ]:
config = Bcf_Extraction_Config(
    Cutoff_At_Kv=40.0,
    Peak_Width_Kev=0.20,
    Bg_Width_Kev=0.10,
    Bg_Offset_Kev=0.25,
    Output_Dir=RAW_OUTPUT_DIR,
)

extractor = Bcf_Element_Extractor(
    Cutoff_At_Kv=config.Cutoff_At_Kv,
    Peak_Width_Kev=config.Peak_Width_Kev,
    Bg_Width_Kev=config.Bg_Width_Kev,
    Bg_Offset_Kev=config.Bg_Offset_Kev,
)

## 5.  Run extraction

Iterate over every discovered BCF file and extract all target elements.

In [ ]:
for bcf_file in bcf_files:
    extractor.Extract_And_Save(
        Bcf_File_Path=bcf_file,
        Target_Elements=TARGET_ELEMENTS,
        Output_Dir=config.Output_Dir,
    )

## 6.  Verify outputs

Enumerate the generated TIFF files and confirm every filename matches
the required `{sample}_{element_line}_raw.tiff` schema.

In [ ]:
import re

tiff_files = sorted(RAW_OUTPUT_DIR.glob("*.tiff"))
print(f"Total TIFFs in raw/: {len(tiff_files)}")

# schema: <sample>_<symbol>_<line>_raw.tiff
pattern = re.compile(r".+_.+_.+_raw\.tiff$")

ok_count = 0
bad_files = []
for tiff_file in tiff_files:
    if pattern.search(tiff_file.name):
        ok_count += 1
    else:
        bad_files.append(tiff_file.name)

print(f"Valid   ({pattern.pattern}): {ok_count}")
print(f"Non-conforming             : {len(bad_files)}")
if bad_files:
    print("Non-conforming files:")
    for name in bad_files:
        print(f"  {name}")

## 7.  Quick visual check

Display the first extracted map that was produced during this session.
This is purely a sanity check — confirm the image is not empty.

In [ ]:
import matplotlib.pyplot as plt
import tifffile

if tiff_files:
    sample_file = tiff_files[0]
    img = tifffile.imread(sample_file)

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(img, cmap="inferno")
    ax.set_title(sample_file.name)
    plt.colorbar(im, ax=ax, label="net intensity")
    plt.tight_layout()
    plt.show()
else:
    print("No TIFF files found in output directory.")